In [1]:
# ========== 导入：Hugging Face / Transformers / PyTorch 相关工具箱 ==========

# 导入 PyTorch：深度学习框架；后面会查 torch.__version__ 与 CUDA 编译信息
import torch
# Colab 专用 userdata 已注释：本环境用环境变量 HF_TOKEN，不走 Colab secrets
#from google.colab import userdata
# 从 huggingface_hub 导入 login：可用 token 登录 Hugging Face Hub（本格未调用，供后续扩展）
from huggingface_hub import login
# 从 transformers 导入 pipeline：一行创建情感分析等推理管线（Pipeline）
from transformers import pipeline
# 图像生成 / 数据集 / 音频写入等依赖本练习未用，保持注释以免多余安装
#from diffusers import DiffusionPipeline
#from datasets import load_dataset
#import soundfile as sf
# 从 IPython.display 导入 Audio：在笔记本里播放音频（本练习未用到）
from IPython.display import Audio


In [2]:
# ========== 检查 Hugging Face Token（环境变量 HF_TOKEN）==========

# 导入标准库 os：读取 Environment Variables（环境变量）
import os

# 从环境变量 HF_TOKEN 读取 Hugging Face 访问令牌（不要把 token 写进代码）
hf_token = os.getenv("HF_TOKEN")

# 简单校验：存在且以 hf_ 开头，才像合法的 HF token 格式
if hf_token and hf_token.startswith("hf_"):
    print("HF_TOKEN OK (parece válido).")
else:
    print("HF_TOKEN no está definido o no tiene formato 'hf_...'.")


HF_TOKEN OK (parece válido).


In [3]:
# ========== 用 shell 魔法检查本机是否连上 NVIDIA GPU ==========

# Jupyter/IPython 魔法：!nvidia-smi 在 shell 里跑 nvidia-smi，返回输出行列表
gpu_info = !nvidia-smi
# 把多行输出拼成一个大字符串，方便后面用 find 搜索关键字
gpu_info = '\n'.join(gpu_info)
# 若输出里含 failed，通常表示没有 GPU 驱动 / 未连接 GPU
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  # 打印完整 nvidia-smi 表格（驱动、显存、进程等）
  print(gpu_info)
  # 进一步看是否是课程期望的 RTX A3000（字符串匹配显卡名）
  if gpu_info.find('A3000') >= 0:
    print("Success - Connected to a NVIDIA RTX A3000")
  else:
    print("NOT CONNECTED TO A NVIDIA RTX A3000")


Thu Jan 29 14:53:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.32                 Driver Version: 581.32         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A3000 Laptop GPU  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P8             10W /   65W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# ========== 在 CPU 上跑默认情感分析 Pipeline ==========

# pipeline("sentiment-analysis")：加载默认情感分类模型
# device=-1 强制走 CPU（本地没有 GPU 或想测 CPU 推理时用）
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device=-1)  # CPU
# 对一句英文做情感打分；返回 label（POSITIVE/NEGATIVE 等）与 score
print(my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!"))


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9993460774421692}]


In [11]:
# ========== 再确认 GPU 状态 + 打印 PyTorch / CUDA 构建信息 ==========

# 再次调用 nvidia-smi（shell 魔法），对照前面单元格的结果
!nvidia-smi
# 打印本机安装的 PyTorch 版本号
print(torch.__version__)
# torch.version.cuda：该 wheel 编译时绑定的 CUDA 版本；None 表示 CPU-only 构建
print("torch CUDA build:", torch.version.cuda)


Thu Jan 29 14:57:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.32                 Driver Version: 581.32         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A3000 Laptop GPU  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8             10W /   65W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# ========== 指定多语言模型，并尝试在 GPU（device=0）上推理 ==========

# model= 指定 Hub 上的多语言情感模型；device=0 表示第一块 CUDA GPU
# 若本机无 GPU，此格可能失败——可改回 device=-1 做 CPU 对比
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=0)
# 对另一句英文推理；该模型常见输出是 1–5 star 星级标签
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
# 打印完整结果列表（含 label / score）
print(result)


Device set to use cpu


[{'label': '3 stars', 'score': 0.3944803476333618}]
